In [1]:
print("HELLO VENV")

HELLO VENV


In [ ]:
import math

import torch
import torch.nn as nn

In [ ]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super(InputEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.scale = math.sqrt(d_model)

    def forward(self, x):
        return self.embedding(x) * self.scale

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super()(PositionalEncoding, self).__init__()

        pos = torch.arange(0, max_len).unsqueeze(1) # shape (d_model, 1)
        i = torch.arange(0, d_model).unsqueeze(0) # shape (1, d_model)
        
        angle_rates = 1 / torch.pow(10000, (2 * (i // 2)) / d_model) # sinusoid frequencies for each pair
        angle_rads = pos * angle_rates


        # Fills the even indices (0, 2, 4,...) with sine values, and the odd indices (1, 3, 5,...) with cosine values.
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(angle_rads[:, 0::2])
        pe[:, 1::2] = torch.cos(angle_rads[:, 1::2])
        pe = pe.unsqueeze(0) # shape (1, max_len, d_model)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        assert (
            self.head_dim * n_heads == d_model
        ), "embed_dim must be divisible by n_heads"

        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.W_out = torch.nn.Linear(d_model, d_model)

    def forward(self, query, key, value, attn_mask=None):

        print("Query size:", query.size())
        B, T, D = query.size()

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        def reshape(x):
            return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        Q, K, V = map(reshape, (Q, K, V))

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        causal_mask = torch.tril(torch.ones(T, T)).to(query.device)
        scores = scores.masked_fill(causal_mask == 0, float('-inf'))

        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask.unsqueeze(1).unsqueeze(2) == 0, float('-inf'))
        
        attn_weights = torch.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)

        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_out(attn_output)
